In [ ]:
!unzip -o project.zip -d .
!pip install "numpy<2.0" pyyaml

import os
os.makedirs("models", exist_ok=True)
print("Code Setup Complete.")

In [ ]:
import os
import shutil

if not os.path.exists("training_content") or not os.path.exists("training_content/val2017"):
    print("Downloading COCO Dataset...")
    if os.path.exists("training_content"): shutil.rmtree("training_content")

    !wget -q http://images.cocodataset.org/zips/val2017.zip
    !mkdir -p training_content
    !unzip -q val2017.zip -d training_content

    if os.path.exists("val2017.zip"): os.remove("val2017.zip")
    print("COCO Dataset Ready at: training_content/val2017")
else:
    print("COCO Dataset already exists.")

if not os.path.exists("models/vgg16.pth"):
    print("Downloading VGG16 Weights...")
    !wget -q https://download.pytorch.org/models/vgg16-397923af.pth -O models/vgg16.pth
    print("VGG Weights Ready.")
else:
    print("VGG Weights already exist.")

In [ ]:
# Multi-style CIN Training
# Trains a single model that handles all 5 curated styles via conditional instance normalization

STYLE_IMAGES = ",".join([
    "styles/starry_night.jpg",
    "styles/great_wave.jpg",
    "styles/girl_pearl.jpg",
    "styles/composition_viii.jpg",
    "styles/water_lilies.jpg",
])
MODEL_NAME = "multistyle"
NUM_STYLES = 5

print(f"Starting CIN Training with {NUM_STYLES} styles...")
print(f"Style images: {STYLE_IMAGES}")

!python -m styleforge.train_cin cin \
    --dataset training_content \
    --style-images {STYLE_IMAGES} \
    --save-model-dir models \
    --save-model-name {MODEL_NAME} \
    --cuda 1 \
    --amp 1 \
    --epochs 4 \
    --limit 10000 \
    --batch-size 4 \
    --image-size 256 \
    --style-size 512 \
    --content-weight 1e5 \
    --style-weight 1e10 \
    --checkpoint-interval 2000

print("CIN Training Finished!")

In [ ]:
from google.colab import files
import os

target_file = f"models/{MODEL_NAME}.pth"

if os.path.exists(target_file):
    print(f"Downloading: {target_file}")
    files.download(target_file)
    print(f"\nModel size: {os.path.getsize(target_file) / 1024 / 1024:.1f} MB")
else:
    print("Model file not found.")

if os.path.exists('models/cin_loss_plot.png'):
    files.download('models/cin_loss_plot.png')

In [ ]:
# Optional: Test style interpolation locally after downloading the model

import torch
from styleforge.cin import CINTransformer

model = CINTransformer(num_styles=NUM_STYLES)
state_dict = torch.load(target_file, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print(f"Model loaded: {NUM_STYLES} styles")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1024 / 1024:.1f}M")

# Test inference with different styles
dummy_input = torch.randn(1, 3, 256, 256)
for style_id in range(NUM_STYLES):
    with torch.no_grad():
        output = model(dummy_input, torch.tensor([style_id]))
    print(f"Style {style_id}: output shape {output.shape}, range [{output.min():.1f}, {output.max():.1f}]")

# Test interpolation between style 0 and style 1
with torch.no_grad():
    interp_output = model.forward_interpolated(dummy_input, style_id_a=0, style_id_b=1, alpha=0.5)
print(f"\nInterpolation (0.5 blend): output shape {interp_output.shape}")
print("All tests passed!")